# 02 — Hartree-Fock Theory

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ppt-2/Ch121a-DFT/blob/main/notebooks/02_hartree_fock.ipynb)

## 🎯 Learning Objectives

By the end of this notebook, you will be able to:
- Explain the Hartree-Fock approximation and Slater determinants
- Perform HF calculations on simple molecules with PySCF
- Monitor SCF convergence and understand the iterative procedure
- Apply Koopmans' theorem to estimate ionization energies
- Understand why HF fails for bond dissociation (static correlation)
- Compare HF and MP2 results for H₂ dissociation

## 1. Theory: Hartree-Fock Approximation

### 1.1 The Variational Principle and Slater Determinants

The exact many-electron wavefunction $\Psi(x_1, x_2, \ldots, x_N)$ is approximated
in HF as a single **Slater determinant** of one-electron spin-orbitals $\chi_i$:

$$\Psi_{HF} = \frac{1}{\sqrt{N!}} \begin{vmatrix} \chi_1(x_1) & \chi_2(x_1) & \cdots & \chi_N(x_1) \\ \chi_1(x_2) & \chi_2(x_2) & \cdots & \chi_N(x_2) \\ \vdots & \vdots & \ddots & \vdots \\ \chi_1(x_N) & \chi_2(x_N) & \cdots & \chi_N(x_N) \end{vmatrix}$$

The Slater determinant **automatically satisfies the Pauli antisymmetry principle**.

### 1.2 The Fock Operator and HF Equations

Applying the variational principle to the Slater determinant yields the
**Hartree-Fock equations** (Roothan equations in the LCAO-MO basis):

$$\hat{F} |\phi_i\rangle = \epsilon_i |\phi_i\rangle$$

The Fock operator is:
$$\hat{F} = \hat{h} + \sum_j [2\hat{J}_j - \hat{K}_j]$$

where:
- $\hat{h}$ = one-electron (core) Hamiltonian (kinetic + nuclear attraction)
- $\hat{J}_j$ = Coulomb operator: $\hat{J}_j \phi_i(1) = \left[\int |\phi_j(2)|^2 r_{12}^{-1} d2\right] \phi_i(1)$
- $\hat{K}_j$ = Exchange operator: $\hat{K}_j \phi_i(1) = \left[\int \phi_j^*(2) \phi_i(2) r_{12}^{-1} d2\right] \phi_j(1)$

### 1.3 SCF Procedure

Because $\hat{F}$ depends on $\{\phi_i\}$, the equations are solved iteratively:

1. **Initial guess** for density matrix $\mathbf{P}$
2. **Build** Fock matrix $F_{\mu\nu} = H_{\mu\nu}^{core} + \sum_{\lambda\sigma} P_{\lambda\sigma} [(\mu\nu|\sigma\lambda) - \frac{1}{2}(\mu\lambda|\sigma\nu)]$
3. **Solve** $\mathbf{FC} = \mathbf{SC}\boldsymbol{\epsilon}$ (generalized eigenvalue problem)
4. **Update** density matrix $P_{\mu\nu} = 2\sum_{i}^{occ} C_{\mu i} C_{\nu i}$
5. **Check convergence** — repeat from step 2 if not converged

### 1.4 Koopmans' Theorem

The negative of the HF orbital energy approximates the ionization energy:
$$IE_i \approx -\epsilon_i$$

This is **Koopmans' theorem** — it assumes the other orbitals do not relax upon ionization
(the frozen orbital approximation). It is exact within HF but neglects orbital relaxation
and correlation effects.

### 1.5 What HF misses: Electron Correlation

HF accounts for Coulomb repulsion only *on average* through the mean-field potential.
It completely misses **electron correlation** — the instantaneous correlated motion
of electrons. The correlation energy is defined as:

$$E_{corr} = E_{exact} - E_{HF} < 0$$

**Static correlation** (near-degenerate configurations) and **dynamic correlation**
(short-range electron-electron cusps) both require post-HF methods.

In [ ]:
# =============================================================================
# Ch121a: Quantum Chemistry & DFT — Notebook 02: Hartree-Fock Theory
# License: GPL-3.0 (https://www.gnu.org/licenses/gpl-3.0.en.html)
# =============================================================================
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pyscf import gto, scf, mp

# ------------------------------------------------------------------
# HF/6-31G* calculations on small molecules
# ------------------------------------------------------------------
molecules = {
    'H2O':  'O 0 0 0; H 0 0.757 -0.469; H 0 -0.757 -0.469',
    'NH3':  'N 0 0 0.116; H 0 0.939 -0.269; H 0.814 -0.469 -0.269; H -0.814 -0.469 -0.269',
    'CO':   'C 0 0 0; O 0 0 1.128',
    'HF':   'H 0 0 0; F 0 0 0.917',
}

results = []
for name, atom_str in molecules.items():
    mol = gto.Mole()
    mol.atom = atom_str
    mol.basis = '6-31g*'
    mol.verbose = 0
    mol.build()
    
    mf = scf.RHF(mol)
    mf.verbose = 0
    e = mf.kernel()
    
    # Dipole moment
    dm = mf.make_rdm1()
    dip = mf.dip_moment(mol, dm, verbose=0)
    dip_mag = np.linalg.norm(dip)
    
    # HOMO/LUMO
    mo_e = mf.mo_energy
    mo_occ = mf.mo_occ
    homo_idx = np.where(mo_occ > 0)[0][-1]
    lumo_idx = homo_idx + 1
    homo_ev = mo_e[homo_idx] * 27.2114
    lumo_ev = mo_e[lumo_idx] * 27.2114
    gap_ev = lumo_ev - homo_ev
    
    results.append({
        'Molecule': name,
        'E_HF (Ha)': round(e, 6),
        'Dipole (D)': round(dip_mag, 3),
        'HOMO (eV)': round(homo_ev, 3),
        'LUMO (eV)': round(lumo_ev, 3),
        'Gap (eV)': round(gap_ev, 3),
    })
    print(f"  {name:5s}  E = {e:.6f} Ha  μ = {dip_mag:.3f} D  HOMO = {homo_ev:.2f} eV")

df = pd.DataFrame(results)
print("\n")
print("HF/6-31G* Results:")
print(df.to_string(index=False))

In [ ]:
# ------------------------------------------------------------------
# SCF Convergence Monitoring
# ------------------------------------------------------------------
# PySCF allows us to inject a callback function that is called
# at the end of each SCF iteration.

energies = []
diis_errors = []

def scf_callback(envs):
    """Called after each SCF iteration."""
    energies.append(envs['e_tot'])
    if 'norm_gorb' in envs:
        diis_errors.append(envs['norm_gorb'])

mol = gto.Mole()
mol.atom = 'O 0 0 0; H 0 0.757 -0.469; H 0 -0.757 -0.469'
mol.basis = '6-31g*'
mol.verbose = 0
mol.build()

mf = scf.RHF(mol)
mf.verbose = 0
mf.callback = scf_callback
mf.kernel()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

iterations = list(range(1, len(energies) + 1))

# Panel 1: Total energy vs iteration
ax1.plot(iterations, energies, 'o-', color='steelblue', linewidth=2, markersize=8)
ax1.axhline(y=energies[-1], color='coral', linestyle='--', alpha=0.7,
            label=f'Converged: {energies[-1]:.8f} Ha')
ax1.set_xlabel('SCF Iteration', fontsize=12)
ax1.set_ylabel('Total HF Energy (Hartree)', fontsize=12)
ax1.set_title('SCF Convergence: Total Energy\nH₂O / 6-31G*', fontsize=12)
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

# Panel 2: Energy change per iteration (log scale)
energy_changes = [abs(energies[i] - energies[i-1]) for i in range(1, len(energies))]
ax2.semilogy(range(2, len(energies)+1), energy_changes, 's-',
             color='seagreen', linewidth=2, markersize=8)
ax2.axhline(y=1e-9, color='coral', linestyle='--', alpha=0.7, label='Convergence threshold')
ax2.set_xlabel('SCF Iteration', fontsize=12)
ax2.set_ylabel('|ΔE| (Hartree)', fontsize=12)
ax2.set_title('SCF Convergence: Energy Change\n(log scale)', fontsize=12)
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()
print(f"\nConverged in {len(energies)} SCF iterations")
print(f"Final energy: {energies[-1]:.10f} Ha")

In [ ]:
%%time
# ------------------------------------------------------------------
# Koopmans' Theorem: IE ≈ -ε_HOMO
# ------------------------------------------------------------------
# Compare predicted ionization energies (-HOMO energy) with experiment

molecules_IE = {
    'H2O':  ('O 0 0 0; H 0 0.757 -0.469; H 0 -0.757 -0.469', 12.62),
    'NH3':  ('N 0 0 0.116; H 0 0.939 -0.269; H 0.814 -0.469 -0.269; H -0.814 -0.469 -0.269', 10.07),
    'CO':   ('C 0 0 0; O 0 0 1.128', 14.01),
    'HF':   ('H 0 0 0; F 0 0 0.917', 16.03),
    'N2':   ('N 0 0 0; N 0 0 1.098', 15.58),
}

koopman_results = []
for name, (atom_str, exp_ie) in molecules_IE.items():
    mol = gto.Mole()
    mol.atom = atom_str
    mol.basis = '6-31g*'
    mol.verbose = 0
    mol.build()
    
    mf = scf.RHF(mol)
    mf.verbose = 0
    mf.kernel()
    
    mo_e = mf.mo_energy
    mo_occ = mf.mo_occ
    homo_idx = np.where(mo_occ > 0)[0][-1]
    comp_ie = -mo_e[homo_idx] * 27.2114
    
    koopman_results.append({
        'Molecule': name,
        'Computed IE (eV)': round(comp_ie, 2),
        'Experimental IE (eV)': exp_ie,
        'Error (eV)': round(comp_ie - exp_ie, 2),
    })

df_k = pd.DataFrame(koopman_results)
print("Koopmans' Theorem: HF/6-31G* Ionization Energies")
print(df_k.to_string(index=False))

mae = np.mean(np.abs(df_k['Error (eV)']))
print(f"\nMean Absolute Error: {mae:.2f} eV")

# Plot
fig, ax = plt.subplots(figsize=(7, 5))
x = np.array(df_k['Experimental IE (eV)'])
y = np.array(df_k['Computed IE (eV)'])
names = list(df_k['Molecule'])

colors_mol = ['steelblue', 'coral', 'seagreen', 'mediumpurple', '#E67E22']
for xi, yi, name, col in zip(x, y, names, colors_mol):
    ax.scatter(xi, yi, color=col, s=120, zorder=5, label=name)

# y=x line (perfect agreement)
lim_min = min(min(x), min(y)) - 0.5
lim_max = max(max(x), max(y)) + 0.5
ax.plot([lim_min, lim_max], [lim_min, lim_max], 'k--', alpha=0.4, label='Perfect agreement')

ax.set_xlabel('Experimental IE (eV)', fontsize=12)
ax.set_ylabel('Computed IE / Koopmans (eV)', fontsize=12)
ax.set_title("Koopmans' Theorem vs Experiment\nHF/6-31G*", fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
%%time
# ------------------------------------------------------------------
# H₂ Bond Dissociation: RHF vs MP2
# ------------------------------------------------------------------
# RHF fails at dissociation because it forces double occupation,
# giving incorrect ionic character (H⁺ + H⁻) at long range.
# MP2 partially corrects this via perturbation theory.

from pyscf import gto, scf, mp

distances = np.linspace(0.5, 4.0, 30)   # Angstroms
e_rhf = []
e_mp2 = []

for d in distances:
    mol = gto.Mole()
    mol.atom = f'H 0 0 0; H 0 0 {d}'
    mol.basis = 'cc-pVDZ'
    mol.verbose = 0
    mol.build()
    
    mf = scf.RHF(mol)
    mf.verbose = 0
    ehf = mf.kernel()
    e_rhf.append(ehf)
    
    # MP2 on top of RHF
    mpt = mp.MP2(mf)
    mpt.verbose = 0
    emp2, _ = mpt.kernel()
    e_mp2.append(ehf + emp2)

e_rhf = np.array(e_rhf)
e_mp2 = np.array(e_mp2)

# Normalize to equilibrium minimum
e_rhf_norm = (e_rhf - e_rhf.min()) * 627.509   # kcal/mol
e_mp2_norm = (e_mp2 - e_mp2.min()) * 627.509

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(distances, e_rhf_norm, 'o-', color='coral', linewidth=2,
        markersize=4, label='RHF/cc-pVDZ')
ax.plot(distances, e_mp2_norm, 's-', color='steelblue', linewidth=2,
        markersize=4, label='MP2/cc-pVDZ')

# Reference: exact 2H dissociation limit
# 2 * E(H atom, UHF) at large distance
ax.axhline(y=0, color='gray', linestyle=':', alpha=0.5)

ax.set_xlabel('H-H Distance (Å)', fontsize=12)
ax.set_ylabel('Relative Energy (kcal/mol)', fontsize=12)
ax.set_title('H₂ Bond Dissociation: RHF vs MP2\n(cc-pVDZ)', fontsize=12)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim(0.4, 4.1)
ax.set_ylim(-5, 80)

# Annotate RHF error
ax.annotate('RHF unphysically\nraises at dissociation\n(static correlation error)',
            xy=(3.5, e_rhf_norm[-3]), xytext=(2.5, 50),
            arrowprops=dict(arrowstyle='->', color='coral'),
            fontsize=9, color='coral')

plt.tight_layout()
plt.show()

print("\nKey insight:")
print("  RHF forces the wavefunction to be a single Slater determinant,")
print("  which cannot properly describe bond breaking.")
print("  At large r(H-H), the true wavefunction should be:")
print("    Ψ ∝ |H↑ H↓⟩ - |H↓ H↑⟩  (purely covalent)")
print("  But RHF gives equal weight to ionic H⁺H⁻ configurations.")

## 🔬 Research Connection

Hartree-Fock theory, despite its limitations, remains fundamental:

- **Starting point for coupled cluster**: CCSD(T), the "gold standard" of quantum chemistry,
  builds on HF via perturbation theory and cluster operators.
- **HF in DFT**: The Kohn-Sham equations have the same structure as HF, but replace
  the exact exchange with an approximate exchange-correlation functional.
- **Orbital analysis**: HF molecular orbitals (sigma, pi, lone pairs) provide chemical
  intuition used daily in organic and organometallic chemistry.
- **Limitations drive innovation**: The failures of HF (correlation energy, dispersion,
  open-shell systems) motivated the entire field of post-HF and DFT methods.

**Notable paper:** Roothaan, C.C.J. (1951). *Rev. Mod. Phys.* **23**, 69 — derived
the matrix HF equations (Roothaan equations) that all modern QC codes implement.

## 📋 Summary

| Concept | Key Result |
|---------|----------|
| Slater determinant | Antisymmetric wavefunction for N fermions; incorporates Pauli principle |
| Fock operator | $\hat{F} = \hat{h} + \hat{J} - \hat{K}$; mean-field treatment of e-e repulsion |
| SCF procedure | Iterative diagonalization until density matrix converges |
| Koopmans' theorem | $\text{IE} \approx -\epsilon_{HOMO}$; error ~0.5-1.5 eV for typical molecules |
| HF correlation energy | $E_{corr} = E_{exact} - E_{HF} \approx$ -0.5 to -1.5 eV/electron pair |
| H₂ dissociation | RHF fails; needs multi-reference methods or UHF |

**Post-HF hierarchy** (increasing accuracy, increasing cost):
HF → MP2 → CCSD → CCSD(T) → FCI

**DFT alternative** (similar cost to HF, better accuracy):
DFT captures ~80-90% of correlation energy implicitly through $E_{xc}$

## 📝 Exercises

1. **Basis set effect on Koopmans'**: Redo the Koopmans' theorem calculation using
   `def2-TZVP` instead of `6-31G*`. Does a larger basis set improve agreement with
   experiment? Why or why not?

2. **SCF starting point**: Try using `mf.init_guess = 'atom'` vs `mf.init_guess = 'huckel'`
   for the water SCF calculation. Does the starting guess affect the converged energy?
   Does it affect the number of iterations?

3. **Open-shell HF**: CO has a singlet ground state. Run UHF (unrestricted HF) on 
   the triplet state of CO using `mol.spin = 2` and `scf.UHF(mol)`.
   What is the energy difference between singlet and triplet?

4. **HOMO-LUMO gap trends**: Compute HF/6-31G* HOMO-LUMO gaps for 
   ethylene (C₂H₄), butadiene (C₄H₆), hexatriene (C₆H₈), and benzene (C₆H₆).
   Plot gap vs number of double bonds. What trend do you observe?

5. **Correlation energy estimate**: Compute RHF and MP2 energies for H₂O with cc-pVTZ.
   What fraction of the correlation energy does MP2 recover (compared to ~-0.37 Ha
   for the experimental correlation energy of water)?